# Notebook 05 - Evaluation finale

Ce notebook orchestre le pipeline complet, sauvegarde les modeles et consolide les resultats finaux.


In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import prepare_raw_datasets
from src.feature_engineering import build_feature_table, temporal_train_test_split, get_feature_columns
from src.project_pipeline import run_full_training
from src.models import train_classifiers, train_rul_models
from src.evaluation import evaluate_classifier, evaluate_regressor
from src.utils import COLOR_PALETTE, RANDOM_STATE, SENSOR_COLUMNS, CRITICAL_RUL_THRESHOLD

plt.style.use("seaborn-v0_8")
sns.set_palette([COLOR_PALETTE["accent"], COLOR_PALETTE["success"], COLOR_PALETTE["info"], COLOR_PALETTE["danger"]])
pd.set_option("display.max_columns", 120)


## 1. Execution du pipeline complet

Cette cellule produit la table de features, les modeles entraines, les predicions et le resume des metriques pour le dashboard.


In [2]:
artifacts = run_full_training(PROJECT_ROOT)
# ... Nettoyage : prints supprimés ...


/home/yambogo-brice/projet-m3-brice/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['hours_since_last_failure' 'volt_lag_1' 'volt_lag_2' 'volt_lag_6'
 'volt_lag_24' 'rotate_lag_1' 'rotate_lag_2' 'rotate_lag_6'
 'rotate_lag_24' 'pressure_lag_1' 'pressure_lag_2' 'pressure_lag_6'
 'pressure_lag_24' 'vibration_lag_1' 'vibration_lag_2' 'vibration_lag_6'
 'vibration_lag_24']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/home/yambogo-brice/projet-m3-brice/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['hours_since_last_failure' 'volt_lag_1' 'volt_lag_2' 'volt_lag_6'
 'volt_lag_24' 'rotate_lag_1' 'rotate_lag_2' 'rotate_lag_6'
 'rotate_lag_24' 'pressure_lag_1' 'pressure_lag_2' 'pressure_lag_6'
 'pressure_lag_24' 'vibration_lag_1' 'vibration_lag_2' 'vibration_lag_6'
 'vibration_lag_24']. At 

/home/yambogo-brice/projet-m3-brice/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['hours_since_last_failure' 'volt_lag_1' 'volt_lag_2' 'volt_lag_6'
 'volt_lag_24' 'rotate_lag_1' 'rotate_lag_2' 'rotate_lag_6'
 'rotate_lag_24' 'pressure_lag_1' 'pressure_lag_2' 'pressure_lag_6'
 'pressure_lag_24' 'vibration_lag_1' 'vibration_lag_2' 'vibration_lag_6'
 'vibration_lag_24']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/home/yambogo-brice/projet-m3-brice/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['hours_since_last_failure' 'volt_lag_1' 'volt_lag_2' 'volt_lag_6'
 'volt_lag_24' 'rotate_lag_1' 'rotate_lag_2' 'rotate_lag_6'
 'rotate_lag_24' 'pressure_lag_1' 'pressure_lag_2' 'pressure_lag_6'
 'pressure_lag_24' 'vibration_lag_1' 'vibration_lag_2' 'vibration_lag_6'
 'vibration_lag_24']. At 

## 2. Resume des metriques

Les metriques de classification et de regression sont affichees avec quatre decimales pour faciliter la comparaison en soutenance.


In [3]:
summary_path = PROJECT_ROOT / "data" / "processed" / "metrics_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
pd.DataFrame(summary["classification"]).T


,0_precision,0_recall,0_f1-score,0_support,1_precision,1_recall,1_f1-score,1_support,accuracy,macro avg_precision,macro avg_recall,macro avg_f1-score,macro avg_support,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,weighted avg_support,roc_auc,average_precision,precision_failure,recall_failure,f1_failure,threshold
dummy,0.00,0.000000,0.000000,103.0,0.608365,1.00000,0.756501,160.0,0.608365,0.304183,0.500000,0.378251,263.0,0.370108,0.608365,0.460229,263.0,0.500000,0.608365,0.608365,1.00000,0.756501,0.500000
logreg,0.75,0.029126,0.056075,103.0,0.613900,0.99375,0.758950,160.0,0.615970,0.681950,0.511438,0.407512,263.0,0.667201,0.615970,0.483679,263.0,0.755643,0.819514,0.613900,0.99375,0.758950,0.703788
rf,0.00,0.000000,0.000000,103.0,0.608365,1.00000,0.756501,160.0,0.608365,0.304183,0.500000,0.378251,263.0,0.370108,0.608365,0.460229,263.0,0.496875,0.606880,0.608365,1.00000,0.756501,0.500000


In [4]:
pd.DataFrame(summary["regression"]).T


,mae,rmse,r2,nasa_score,critical_zone_mae
dummy_rul,525.691667,525.716157,-10732.366452,7.545235e+22,525.966030
rf_rul,440.622812,474.998147,-8761.280233,6.149499e+43,441.585987
svr_rul,402.091442,402.123323,-6278.887068,3.232910e+17,402.365304


## 3. Verification des artefacts

Les modeles `joblib`, les predictions flotte et les tableaux de metriques doivent etre presents dans les dossiers de sortie.


In [5]:
sorted(str(path.relative_to(PROJECT_ROOT)) for path in PROJECT_ROOT.rglob("*") if path.is_file())[:40]


['README.md',
 '__pycache__/generate_notebooks.cpython-312.pyc',
 '__pycache__/run_pipeline.cpython-312.pyc',
 'dashboard/__pycache__/app.cpython-312.pyc',
 'dashboard/app.py',
 'data/processed/all_predictions.csv',
 'data/processed/feature_table.csv',
 'data/processed/fleet_predictions.csv',
 'data/processed/fleet_snapshot.csv',
 'data/processed/metrics_summary.json',
 'data/raw/PdM_errors.csv',
 'data/raw/PdM_failures.csv',
 'data/raw/PdM_machines.csv',
 'data/raw/PdM_maint.csv',
 'data/raw/PdM_telemetry.csv',
 'data/raw/ai4i2020.csv',
 'generate_notebooks.py',
 'models/classifier_rf.joblib',
 'models/classifier_xgb.joblib',
 'models/regressor_rul.joblib',
 'notebooks/01_EDA.ipynb',
 'notebooks/02_feature_engineering.ipynb',
 'notebooks/03_classification_panne.ipynb',
 'notebooks/04_regression_RUL.ipynb',
 'notebooks/05_evaluation_finale.ipynb',
 'notebooks/06_finalisation_projet.ipynb',
 'rapport/rapport_M3.md',
 'rapport/rapport_M3.pdf',
 'requirements.txt',
 'run_pipeline.py',
 's

## 4. Conclusion generale

Le pipeline tourne de bout en bout et alimente directement le dashboard Streamlit. Le projet est donc pret pour la demonstration et la redaction finale.
